In [5]:
def write_cpw_array_cif(gnd_width, slot_widths, sig_width, length, 
                        edge_to_edge_separation,
                        metal_layer=1, slot_layer=3,
                        filename="cpw_array.cif"):
    """
    Generates a CIF file for an array of CPW structures with varying slot widths.

    Inputs:
    - gnd_width: Width of the ground planes (µm)
    - slot_widths: A list of slot widths/gaps to iterate through (µm), e.g., [5, 10, 15]
    - sig_width: Width of the center signal line (µm)
    - length: Length of the CPW lines (µm)
    - edge_to_edge_separation: Distance between the outer edges of adjacent CPWs (µm)
    - unit_scale: Conversion factor to database units (1000 converts µm to nm)
    """

    def to_int(x):
        
        return int(x * 100)

    def rect(x1, y1, x2, y2):
        x1i, y1i = to_int(x1), to_int(y1)
        x2i, y2i = to_int(x2), to_int(y2)

        width  = x2i - x1i
        height = y2i - y1i
        cx     = (x1i + x2i) // 2
        cy     = (y1i + y2i) // 2

        return f"B {width} {height} {cx} {cy};\n"

    with open(filename, "w") as f:
        f.write("(Made by Hrithik Singla);\n")
        f.write("(MDHS Lab, CeNSE, IISc Bengaluru);\n")
        
        # Cleaned up the triple quotes to prevent indentation spaces in the CIF
        f.write("(Layer names);\n")
        f.write("L L0; (CleWin: 0 Outline/06808000 06808000);\n")
        f.write("L L1; (CleWin: 1 Au/0f02e8fe 0f02e8fe);\n")
        f.write("L L2; (CleWin: 2 dielet_align/0f800080 0f800080);\n")
        f.write("L L3; (CleWin: 3 gaps/0fb9bcc6 0fb9bcc6);\n")
        f.write("L L4; (CleWin: 4 markings/0f1383f7 0f1383f7);\n")
        f.write("L L5; (CleWin: 5 pillars/0f51ee36 0f51ee36);\n")
        f.write("L L6; (CleWin: 6 wafer_align/0f0413f9 0f0413f9);\n")
        
        # Setting standard scaling numerator/denominator for a nm database
        f.write("DS1 1 1;\n")
        f.write("9 MainSymbol;\n")

        # The X-cursor keeps track of where the next CPW's left edge should begin
        current_left_x = 0.0 

        for i, slot_width in enumerate(slot_widths):
            # 1. Calculate the total bounding box width of this specific CPW
            total_cpw_width = sig_width + (2 * slot_width) + (2 * gnd_width)
            
            # 2. Determine the center X coordinate for this CPW
            xc = current_left_x + (total_cpw_width / 2.0)

            # f.write(f"\n(--- CPW {i+1}: Slot={slot_width}um ---)\n")

            # --- Write Metal Layer (Signal and Grounds) ---
            f.write(f"L L{metal_layer};\n")

            # Signal
            f.write(rect(xc - sig_width/2, 0, 
                         xc + sig_width/2, length))

            # Left Ground
            f.write(rect(xc - sig_width/2 - slot_width - gnd_width, 0,
                         xc - sig_width/2 - slot_width, length))
            
            # Right Ground
            f.write(rect(xc + sig_width/2 + slot_width, 0,
                         xc + sig_width/2 + slot_width + gnd_width, length))

            # --- Write Slot Layer (Gaps) ---
            f.write(f"L L{slot_layer};\n")

            # Left Slot
            f.write(rect(xc - sig_width/2 - slot_width, 0,
                         xc - sig_width/2, length))
            
            # Right Slot
            f.write(rect(xc + sig_width/2, 0,
                         xc + sig_width/2 + slot_width, length))

            # --- Write filler rectangles between adjacent CPWs ---
            if i < len(slot_widths) - 1:
                f.write(rect(current_left_x + total_cpw_width, 0, 
                             current_left_x + total_cpw_width + edge_to_edge_separation, length))

            # 3. Advance the X cursor for the next iteration
            current_left_x += total_cpw_width + edge_to_edge_separation

        # --- Write Bounding Box around the entire array ---
        # The total width of the array is the current X cursor minus the final separation gap added in the last loop
        total_array_width = current_left_x - edge_to_edge_separation
        bbox_thickness = 200.0

        f.write(f"L L{slot_layer};\n")
        
        # Bottom Edge
        f.write(rect(0, -bbox_thickness, total_array_width, 0))
        # Top Edge
        f.write(rect(0, length, total_array_width, length + bbox_thickness))
        # Left Edge (extends past top/bottom to make solid corners)
        f.write(rect(-bbox_thickness, -bbox_thickness, 0, length + bbox_thickness))
        # Right Edge (extends past top/bottom to make solid corners)
        f.write(rect(total_array_width, -bbox_thickness, total_array_width + bbox_thickness, length + bbox_thickness))

        f.write("DF;\n")
        f.write("C 1;\n")
        f.write("E\n")

    print(f"✅ CIF file '{filename}' created successfully with {len(slot_widths)} CPW lines.")

In [7]:
# Example execution:
write_cpw_array_cif(gnd_width=180, slot_widths=[ 25.5, 27.5, 29.5, 31.5], sig_width=40, length=2500, edge_to_edge_separation=200)

✅ CIF file 'cpw_array.cif' created successfully with 4 CPW lines.
